# 🍀 YOTSUBA FINDER — 自分で撮った写真でモデルを試す

学習済みモデル（`best.pt`）を、自分で撮った写真で試すノートブックです。次の 2 通りで検出して、結果を比べます。

| 方法 | やり方 | ねらい |
|---|---|---|
| **A. 全体** | 写真全体を 960px に縮めて 1 回だけ検出 | アプリの通常の検出と同じ条件 |
| **B. タイル** | 写真を重なりのある 2×2 のタイルに分け、タイルごとに検出してからまとめる | 群生地で、小さく写った葉を見つけやすくなるかを確かめる |

結果は、四つ葉・三つ葉の数の表と、枠を描いた画像で確認します。
枠は YOLO 形式の txt としても保存するので、Roboflow に読み込んで「下書きのラベル」として直せます。

> 実行前に「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4 など）** を選んでください（CPU でも動きますが遅くなります）。

## 1. 設定

ここの値を変えるだけで、場所やしきい値を変えられます。

- **`BEST_SRC`**：コピー元の `best.pt`。`None` のままなら、`runs/` の中でいちばん新しいものを自動で探します
- **`CONF`**：信頼度のしきい値。モデルの自信がこれ以上の枠だけを残します。上げると間違いは減りますが、見逃しが増えます
- **`CONF_LIST`**：しきい値ごとに検出数を比べる表で使う値
- **`TILE_FRAC`**：タイル 1 枚の大きさ（元の写真の幅・高さに対する割合）。0.6 なら、タイル同士の重なりは写真の 20%（タイルの約 33%）
- **`MERGE_IOU`**：タイルの結果をまとめるとき、どれだけ重なった枠を「同じ葉」とみなすか（0〜1）
- **`MERGE_SMALLER_OVERLAP`**：小さいほうの枠の、何割が大きいほうと重なっていたら「同じ葉」とみなすか（0〜1）。タイルの境目で、同じ葉に枠が 2 つ付くのを防ぐ

In [ ]:
from pathlib import Path

# ── 場所 ──
DRIVE_DIR = Path('/content/drive/MyDrive/yotsuba')  # ドライブ上の作業フォルダ
RUNS_DIR = DRIVE_DIR / 'runs'                       # 学習結果のフォルダ
BEST_SRC = None   # コピー元の best.pt。例: DRIVE_DIR / 'runs/yolo11n_960/weights/best.pt'
BEST_PT = DRIVE_DIR / 'best.pt'                     # このノートブックで使う重み
PHOTO_DIR = DRIVE_DIR / 'own_photos'                # 自分で撮った写真（JPEG）
OUT_DIR = DRIVE_DIR / 'pred_out'                    # 結果の保存先

# ── 検出の設定 ──
IMGSZ = 960                   # 学習と同じ画像サイズ
CONF = 0.25                   # 信頼度のしきい値（画像の保存・数の表で使う）
CONF_LIST = [0.25, 0.4, 0.5]  # 検出数を比べるしきい値

# ── タイル（方法 B）の設定 ──
TILE_FRAC = 0.6       # タイル 1 枚 = 元の写真の 60%
MERGE_IOU = 0.5       # IoU がこの値を超えた枠は、同じ葉としてまとめる
MERGE_SMALLER_OVERLAP = 0.6  # 小さいほうの枠に対する重なりがこの値以上の枠も、同じ葉としてまとめる
AGNOSTIC_NMS = True   # True: クラスが違っても重なった枠はまとめる（理由はセクション 5 の説明を参照）

CLASS_NAMES = {0: 'four_leaf', 1: 'three_leaf'}

# タイルの重なりが 20% 以上あるかを確かめる（2 枚のタイルの重なり = 2 × TILE_FRAC − 1）
# （小数の計算誤差で 0.19999… になるのを防ぐため、丸めてから比べる）
TILE_OVERLAP = round(2 * TILE_FRAC - 1, 6)
assert TILE_OVERLAP >= 0.2, f'タイルの重なりが {TILE_OVERLAP:.0%} しかありません。TILE_FRAC を 0.6 以上にしてください'
print(f'タイルの重なり: 写真の {TILE_OVERLAP:.0%}（タイルの {TILE_OVERLAP / TILE_FRAC:.0%}）')

## 2. ライブラリのインストール

YOLO を動かす **ultralytics** を入れます。

In [ ]:
!pip install -q ultralytics

## 3. Google ドライブの接続

許可を求める画面が出たら、自分のアカウントを選んで許可してください。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 4. best.pt を `MyDrive/yotsuba/best.pt` にコピーする

学習のノートブックは、`runs/<結果フォルダ>/weights/best.pt` に重みを保存しています。
それを `MyDrive/yotsuba/best.pt` にコピーして、このノートブックではそれを使います。

すでに `best.pt` を自分で置いた場合は、このセルを飛ばしてかまいません（上書きされます）。

In [ ]:
import shutil

if BEST_SRC is None:
    # runs/ の下の */weights/best.pt から、いちばん新しいものを選ぶ
    candidates = sorted(RUNS_DIR.glob('*/weights/best.pt'), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f'{RUNS_DIR} の下に weights/best.pt が見つかりません。BEST_SRC にパスを書いてください。')
    BEST_SRC = candidates[-1]

shutil.copy(BEST_SRC, BEST_PT)
print(f'コピーしました: {BEST_SRC}\n            → {BEST_PT}')

## 5. 検出のための関数を用意する

このセルでは、あとで使う「道具」（関数）を作るだけで、まだ検出はしません。

### 方法 B（タイル）のしくみ

```
┌──────────┬───┬──────────┐      写真を 4 枚のタイルに分ける。
│  左上    │重な│  右上    │      各タイルは写真の幅・高さの 60%。
│          │ り │          │      真ん中の帯（写真の 20%）は
├──────────┼───┼──────────┤      2 枚以上のタイルに重なって入る。
│ 重なり   │   │          │
├──────────┼───┼──────────┤
│  左下    │   │  右下    │
└──────────┴───┴──────────┘
```

1. タイルごとに 960px で検出する（タイルは写真より小さいので、葉が大きく写る）
2. タイルの中の座標に、タイルの左上の位置を足して、元の写真の座標に戻す
3. 重なりの部分では同じ葉が 2 回以上見つかるので、**NMS**（重なった枠のうち、自信のいちばん高い 1 つだけを残す処理）でまとめる

**同じ葉とみなす基準は 2 つ**あり、どちらかに当てはまればまとめます。
- **IoU**（重なった面積 ÷ 2 つを合わせた面積）が `MERGE_IOU` を超える
- **小さいほうの枠に対する重なり**（重なった面積 ÷ 小さいほうの面積）が `MERGE_SMALLER_OVERLAP` 以上

2 つ目が必要な理由：タイルの境目で葉が切れると、同じ葉に「全体の枠」（隣のタイル）と「切れた一部の枠」が付きます。
一部の枠は小さいので IoU は小さくなり、IoU だけではまとまりません。でも、一部の枠はほぼ全部が全体の枠の中に入っているので、2 つ目の基準でまとまります。

**`AGNOSTIC_NMS = True` にしている理由**：タイルの端で四つ葉が切れると、葉が 3 枚しか写らず、三つ葉と判定されることがあります。
クラスが違っても重なった枠はまとめ、自信の高いほう（たいていは葉が全部写ったタイルの結果）を残すようにしています。

In [ ]:
import math
import numpy as np
from PIL import Image, ImageOps, ImageDraw, ImageFont

def load_photo(path):
    """JPEG を読み込み、EXIF の向き情報どおりに回転した RGB 画像を返す（スマホの写真は向き情報を持っていることが多い）。"""
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert('RGB')

def to_numpy(x):
    """YOLO の結果（GPU 上の tensor）を numpy の配列にする。"""
    return x.cpu().numpy() if hasattr(x, 'cpu') else np.asarray(x)

def boxes_of(result):
    """YOLO の結果から、枠の座標（左上 x, y・右下 x, y）・信頼度・クラス番号を取り出す。"""
    b = result.boxes
    xyxy = to_numpy(b.xyxy).astype(float).reshape(-1, 4)
    conf = to_numpy(b.conf).astype(float).reshape(-1)
    cls = to_numpy(b.cls).astype(int).reshape(-1)
    return xyxy, conf, cls

def to_bgr(rgb_array):
    """YOLO に numpy の画像を渡すときは BGR の順番にする必要がある。"""
    return np.ascontiguousarray(rgb_array[:, :, ::-1])

def box_iou(box, boxes):
    """1 つの枠と、複数の枠との重なり具合（IoU = 重なった面積 ÷ 2 つを合わせた面積）を計算する。"""
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda b: (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (area(box) + area(boxes) - inter + 1e-9)

def overlap_of_smaller(box, boxes):
    """1 つの枠と、複数の枠との「小さいほうの枠に対する重なり」（重なった面積 ÷ 小さいほうの面積）を計算する。"""
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda b: (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (np.minimum(area(box), area(boxes)) + 1e-9)

def nms(xyxy, conf, cls, iou_thr=MERGE_IOU, smaller_thr=MERGE_SMALLER_OVERLAP, agnostic=AGNOSTIC_NMS):
    """NMS: 自信の高い順に枠を残し、それと「同じ葉」とみなした枠を消していく。残った枠の番号を返す。
    同じ葉とみなすのは、IoU が iou_thr を超えるとき、または、小さいほうの枠に対する重なりが smaller_thr 以上のとき。"""
    order = np.argsort(-conf)
    keep = []
    while order.size:
        best, rest = order[0], order[1:]
        keep.append(best)
        overlapped = (box_iou(xyxy[best], xyxy[rest]) > iou_thr) | (overlap_of_smaller(xyxy[best], xyxy[rest]) >= smaller_thr)
        if not agnostic:
            overlapped &= cls[rest] == cls[best]  # 同じクラス同士だけをまとめる
        order = rest[~overlapped]
    return np.array(keep, dtype=int)

def tile_windows(width, height, frac=TILE_FRAC):
    """2×2 のタイルの範囲（左上 x, y・右下 x, y）を 4 つ返す。"""
    # 切り上げにして、重なりが必ず写真の (2 × frac − 1) 以上になるようにする
    tw, th = math.ceil(width * frac), math.ceil(height * frac)
    xs, ys = [0, width - tw], [0, height - th]
    return [(x, y, x + tw, y + th) for y in ys for x in xs]

def detect_full(model, photo, conf):
    """方法 A: 写真全体を 1 回で検出する。"""
    result = model.predict(to_bgr(np.array(photo)), imgsz=IMGSZ, conf=conf, verbose=False)[0]
    return boxes_of(result)

def detect_tiles(model, photo, conf):
    """方法 B: 2×2 のタイルごとに検出し、元の座標に戻してから NMS でまとめる。"""
    rgb = np.array(photo)
    windows = tile_windows(*photo.size)
    crops = [to_bgr(rgb[y0:y1, x0:x1]) for x0, y0, x1, y1 in windows]
    results = model.predict(crops, imgsz=IMGSZ, conf=conf, verbose=False)

    all_xyxy, all_conf, all_cls = [], [], []
    for (x0, y0, _, _), result in zip(windows, results):
        xyxy, c, k = boxes_of(result)
        all_xyxy.append(xyxy + [x0, y0, x0, y0])  # タイルの中の座標 → 元の写真の座標
        all_conf.append(c)
        all_cls.append(k)
    xyxy, c, k = np.concatenate(all_xyxy), np.concatenate(all_conf), np.concatenate(all_cls)

    keep = nms(xyxy, c, k)  # 重なりの部分で 2 回見つかった枠を 1 つにまとめる
    return xyxy[keep], c[keep], k[keep]

def above(dets, thr):
    """信頼度がしきい値以上の枠だけを残す。"""
    xyxy, conf, cls = dets
    m = conf >= thr
    return xyxy[m], conf[m], cls[m]

def count(dets):
    """クラスごとの枠の数を数える。"""
    return {name: int((dets[2] == i).sum()) for i, name in CLASS_NAMES.items()}

COLORS = {0: (255, 196, 0), 1: (0, 160, 255)}  # 四つ葉 = 金色、三つ葉 = 青

def draw(photo, dets, windows=None):
    """写真に枠を描き込んだ画像を返す（windows を渡すと、タイルの境目も点線で描く）。"""
    im = photo.copy()
    d = ImageDraw.Draw(im)
    w = max(3, round(max(im.size) / 300))  # 写真の大きさに合わせた線の太さ
    try:
        font = ImageFont.truetype('DejaVuSans-Bold.ttf', w * 6)
    except OSError:
        font = ImageFont.load_default()
    for x0, y0, x1, y1 in windows or []:
        for i in range(0, round(x1 - x0), w * 8):  # タイルの範囲を白い点線で描く
            d.line([(x0 + i, y0), (x0 + i + w * 4, y0)], fill='white', width=w // 2 + 1)
            d.line([(x0 + i, y1 - 1), (x0 + i + w * 4, y1 - 1)], fill='white', width=w // 2 + 1)
        for i in range(0, round(y1 - y0), w * 8):
            d.line([(x0, y0 + i), (x0, y0 + i + w * 4)], fill='white', width=w // 2 + 1)
            d.line([(x1 - 1, y0 + i), (x1 - 1, y0 + i + w * 4)], fill='white', width=w // 2 + 1)
    for (x0, y0, x1, y1), c, k in zip(*dets):
        d.rectangle([x0, y0, x1, y1], outline=COLORS[int(k)], width=w)
        d.text((x0 + w, y0 + w), f'{CLASS_NAMES[int(k)]} {c:.2f}', fill=COLORS[int(k)], font=font)
    return im

def yolo_lines(dets, width, height):
    """枠を YOLO 形式の行（クラス番号 中心x 中心y 幅 高さ、0〜1 に正規化）にする。"""
    lines = []
    for (x0, y0, x1, y1), _, k in zip(*dets):
        x0, x1 = np.clip([x0, x1], 0, width)   # 写真の外にはみ出した分は切り取る
        y0, y1 = np.clip([y0, y1], 0, height)
        cx, cy = (x0 + x1) / 2 / width, (y0 + y1) / 2 / height
        bw, bh = (x1 - x0) / width, (y1 - y0) / height
        lines.append(f'{int(k)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    return lines

### 5-1. まとめ方の確認（合成した枠でのテスト）

上の `nms` が意図どおりに動くかを、作り物の枠で確かめます（モデルは使わないので、すぐ終わります）。
「✅」と表示されれば OK です。`MERGE_IOU` や `MERGE_SMALLER_OVERLAP` を変えたときも、ここで動きを確かめられます。

In [ ]:
def check(name, boxes, conf, cls, expected_keep, **kw):
    """作り物の枠で nms を動かし、残った枠の番号が expected_keep と同じかを確かめる。"""
    keep = sorted(int(i) for i in nms(np.array(boxes, float), np.array(conf, float), np.array(cls), **kw))
    assert keep == expected_keep, f'{name}: 残った枠 {keep}（期待 {expected_keep}）'
    print(f'✅ {name}')

# 枠は [左上x, 左上y, 右下x, 右下y]。クラスは 0 = four_leaf、1 = three_leaf
whole, cut = [100, 100, 200, 200], [100, 100, 140, 200]   # 同じ四つ葉の「全体の枠」と「境目で切れた一部の枠」（IoU は 0.4）
check('境目の同じ四つ葉の 2 つの枠は、1 つになる（スコアの高いほうが残る）', [whole, cut], [0.9, 0.7], [0, 0], [0])
check('切れた一部のほうがスコアが高ければ、そちらが残る', [whole, [150, 100, 200, 200]], [0.6, 0.8], [0, 0], [1])
check('クラスが違っても、同じ物なら 1 つになる', [whole, [130, 100, 200, 190]], [0.85, 0.6], [0, 1], [0])
check('離れた別の葉の枠は、両方残る', [[0, 0, 100, 100], [300, 300, 400, 400]], [0.9, 0.8], [0, 1], [0, 1])
check('隣り合って少しだけ重なる別の葉の枠は、両方残る', [[0, 0, 100, 100], [90, 0, 190, 100]], [0.9, 0.8], [1, 1], [0, 1])
check('「小さいほうに対する重なり」を使わない設定では、境目の 2 つの枠は両方残る', [whole, cut], [0.9, 0.7], [0, 0], [0, 1], smaller_thr=1.01)

## 6. モデルと写真を読み込んで、検出する

`own_photos/` の中の JPEG（`.jpg` / `.jpeg`）をすべて読み込み、方法 A と方法 B で検出します。

しきい値を変えて比べられるように、ここでは `CONF_LIST` と `CONF` の中でいちばん低い値で検出しておき、
あとから「しきい値以上の枠だけを残す」形で絞り込みます（しきい値ごとに検出し直す必要がありません）。

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST_PT)

photo_paths = sorted(p for p in PHOTO_DIR.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg'))
if not photo_paths:
    raise FileNotFoundError(f'{PHOTO_DIR} に JPEG がありません。')
print(f'写真: {len(photo_paths)} 枚')

MIN_CONF = min(CONF_LIST + [CONF])  # いちばん低いしきい値で検出しておく
photos, results_a, results_b = {}, {}, {}
for path in photo_paths:
    photo = load_photo(path)
    photos[path.stem] = photo
    results_a[path.stem] = detect_full(model, photo, MIN_CONF)
    results_b[path.stem] = detect_tiles(model, photo, MIN_CONF)
    print(f'  {path.name}  {photo.size[0]}×{photo.size[1]}')

## 7. しきい値ごとの検出数を比べる

全部の写真を合わせた検出数を、しきい値ごとに並べます。

**読み方**
- しきい値を上げるほど、数は減ります
- 上げても**ほとんど減らない**枠は、モデルが自信を持っている枠です
- 上げると**大きく減る**場合は、自信の低い枠（間違いかもしれない枠）が多いということです。セクション 9 の画像と見比べて、どのしきい値がちょうど良いかを考えます

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for thr in CONF_LIST:
    total = {'しきい値': thr}
    for label, results in [('A 全体', results_a), ('B タイル', results_b)]:
        for name in CLASS_NAMES.values():
            total[f'{label}: {name}'] = sum(count(above(r, thr))[name] for r in results.values())
    rows.append(total)
display(pd.DataFrame(rows).set_index('しきい値'))

## 8. 写真ごとの四つ葉・三つ葉の数

しきい値 `CONF` での、写真ごとの数です。
B（タイル）のほうが多ければ、タイルに分けたことで小さな葉を新しく見つけられた可能性があります（間違いが増えただけのこともあるので、画像でも確かめます）。

In [ ]:
rows = []
for stem in photos:
    a, b = count(above(results_a[stem], CONF)), count(above(results_b[stem], CONF))
    rows.append({'写真': stem,
                 'A 四つ葉': a['four_leaf'], 'A 三つ葉': a['three_leaf'],
                 'B 四つ葉': b['four_leaf'], 'B 三つ葉': b['three_leaf']})
table = pd.DataFrame(rows).set_index('写真')
table.loc['合計'] = table.sum()
print(f'しきい値 CONF = {CONF}')
display(table)

## 9. 結果を並べて表示し、画像を保存する

左が **A（全体）**、右が **B（タイル）** です。右の白い点線はタイルの範囲です。
枠の色は **金色 = 四つ葉**、**青 = 三つ葉** で、ラベルの数字は信頼度です（確認用で、アプリには表示しません）。

画像は `MyDrive/yotsuba/pred_out/annotated/` に `<写真の名前>_A_full.jpg` と `<写真の名前>_B_tile.jpg` として保存します。

In [ ]:
import matplotlib.pyplot as plt

ANNOTATED_DIR = OUT_DIR / 'annotated'
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

for stem, photo in photos.items():
    img_a = draw(photo, above(results_a[stem], CONF))
    img_b = draw(photo, above(results_b[stem], CONF), windows=tile_windows(*photo.size))
    img_a.save(ANNOTATED_DIR / f'{stem}_A_full.jpg', quality=90)
    img_b.save(ANNOTATED_DIR / f'{stem}_B_tile.jpg', quality=90)

    a, b = count(above(results_a[stem], CONF)), count(above(results_b[stem], CONF))
    w, h = photo.size
    fig, axes = plt.subplots(1, 2, figsize=(16, 8 * h / w + 0.6))
    # グラフの文字は日本語だと文字化けするので、英語で書く
    for ax, img, title in [(axes[0], img_a, f'A full: four_leaf {a["four_leaf"]} / three_leaf {a["three_leaf"]}'),
                           (axes[1], img_b, f'B tile: four_leaf {b["four_leaf"]} / three_leaf {b["three_leaf"]}')]:
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(stem)
    plt.tight_layout()
    plt.show()

print('保存先:', ANNOTATED_DIR)

## 10. 枠を YOLO 形式の txt で保存する（Roboflow で直す用）

方法ごとに、Roboflow にそのまま読み込める形でフォルダを作ります。

```
pred_out/
├── annotated/        ← セクション 9 の枠付き画像
├── roboflow_A/       ← 方法 A の結果
│   ├── data.yaml
│   ├── images/       ← 向きを直した写真（EXIF の回転を反映済み）
│   └── labels/       ← 1 行 = 1 つの枠「クラス番号 中心x 中心y 幅 高さ」（0〜1）
└── roboflow_B/       ← 方法 B の結果（同じ形）
```

**`images/` に写真を入れ直している理由**：スマホの写真は「向き情報（EXIF）」で回転して表示されることがあります。
枠の座標は回転を反映した後の写真に合わせているので、元の写真ではなく、`images/` の写真と組み合わせてアップロードしてください。

**Roboflow への読み込み方**：Roboflow のプロジェクトで「Upload Data」を開き、`roboflow_A`（または `roboflow_B`）フォルダごとドラッグします。
枠は下書きとして入るので、間違っている枠を消したり、見逃した葉に枠を足したりして直します。

In [ ]:
for method, results in [('A', results_a), ('B', results_b)]:
    folder = OUT_DIR / f'roboflow_{method}'
    (folder / 'images').mkdir(parents=True, exist_ok=True)
    (folder / 'labels').mkdir(parents=True, exist_ok=True)

    for stem, photo in photos.items():
        photo.save(folder / 'images' / f'{stem}.jpg', quality=95)  # 向きを直した写真
        lines = yolo_lines(above(results[stem], CONF), *photo.size)
        (folder / 'labels' / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))

    # Roboflow がクラス名を知るための data.yaml
    names = '\n'.join(f'  {i}: {n}' for i, n in CLASS_NAMES.items())
    (folder / 'data.yaml').write_text(f'train: images\nval: images\nnc: {len(CLASS_NAMES)}\nnames:\n{names}\n')
    print(f'保存しました: {folder}（{len(photos)} 枚）')

# 1 枚目のラベルの中身を見本として表示する
first = next(iter(photos))
print(f'\n例: roboflow_B/labels/{first}.txt')
print((OUT_DIR / 'roboflow_B' / 'labels' / f'{first}.txt').read_text() or '（枠なし）')

## 11. おわりに

| できたもの | 場所 |
|---|---|
| 枠付きの画像 | `MyDrive/yotsuba/pred_out/annotated/` |
| Roboflow 用（方法 A） | `MyDrive/yotsuba/pred_out/roboflow_A/` |
| Roboflow 用（方法 B） | `MyDrive/yotsuba/pred_out/roboflow_B/` |

**比べるときのヒント**
- B で四つ葉の見逃しが減っているなら、アプリの「群生地スキャン」でタイル分割を使う価値があります（ただし検出に約 4 倍の時間がかかります）
- B で間違いばかり増えているなら、`CONF` を上げるか、`MERGE_IOU` や `MERGE_SMALLER_OVERLAP` を下げて（例: 0.3 / 0.5）まとめやすくして試します
- 直したラベルを学習データに足すと、モデルが自分の撮影環境に強くなります